# Suspend new cards with already-reviewed Lemma (AnkiConnect)

This notebook:
- builds a set of lemmas from `is:review` notes
- finds `is:new` cards
- if a new card's note has an identical lemma, suspends that card
- tags both the suspended new notes and the review source notes
- writes sibling links into `Link` field using `[Title|nid...]`

Run first with `DRY_RUN = True`.

In [4]:
import json
import re
import urllib.request
from collections import defaultdict

ANKI_CONNECT_URL = "http://127.0.0.1:8765"
ANKI_CONNECT_VERSION = 6

DECK_SCOPE = 'deck:"Japanese Media"'
REVIEW_QUERY = f"{DECK_SCOPE} is:review"
NEW_QUERY = f"{DECK_SCOPE} is:new"

LEMMA_FIELD_PREFERRED = "Lemma"
LEMMA_FIELD_ALIASES = ["Lemma", "lemma", "Word", "Expression"]
SUBTITLE_FIELD_PREFERRED = "Subtitle"
SUBTITLE_FIELD_ALIASES = ["Subtitle", "subtitle", "Sentence", "Context"]
LINK_FIELD = "Link"

TAG_SUSPENDED_DUPLICATE = "~suspended_duplicate_lemma_review"
TAG_UNIQUE_REVIEW_NOTE = "~unique_note_with_suspended_siblings"
NEW_ELIGIBLE_QUERY = f"{NEW_QUERY} -is:suspended -tag:{TAG_SUSPENDED_DUPLICATE}"
DRY_RUN = False

TAG_RE = re.compile(r"<[^>]+>")

def invoke(action, **params):
    payload = json.dumps({"action": action, "version": ANKI_CONNECT_VERSION, "params": params}).encode("utf-8")
    req = urllib.request.Request(ANKI_CONNECT_URL, data=payload, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        body = json.loads(resp.read().decode("utf-8"))
    if body.get("error"):
        raise RuntimeError(f"AnkiConnect error in {action}: {body['error']}")
    return body.get("result")

def chunked(items, size=500):
    for i in range(0, len(items), size):
        yield items[i:i + size]

def get_notes_info(note_ids):
    out = []
    for batch in chunked(note_ids, 500):
        out.extend(invoke("notesInfo", notes=batch))
    return out

def get_cards_info(card_ids):
    out = []
    for batch in chunked(card_ids, 500):
        out.extend(invoke("cardsInfo", cards=batch))
    return out

def resolve_field_name(fields_dict, preferred, aliases):
    names = list(fields_dict.keys())
    if preferred in fields_dict:
        return preferred
    folded = {n.casefold(): n for n in names}
    if preferred.casefold() in folded:
        return folded[preferred.casefold()]
    for alias in aliases:
        if alias in fields_dict:
            return alias
        if alias.casefold() in folded:
            return folded[alias.casefold()]
    return None

def normalize_lemma(raw):
    text = TAG_RE.sub("", raw or "")
    return re.sub(r"\s+", " ", text).strip()

def normalize_subtitle(raw):
    text = TAG_RE.sub("", raw or "")
    text = re.sub(r"\s+", " ", text).strip()
    return text[:120]

def sanitize_link_title(text):
    return (text or "").replace("[", "(").replace("]", ")").replace("|", "/").strip()

def make_link(title, nid):
    return f"[{sanitize_link_title(title)}|nid{nid}]"

print("Connected AnkiConnect version:", invoke("version"))

Connected AnkiConnect version: 6


In [5]:
# 1) Build set + map of lemmas from review notes
review_note_ids = invoke("findNotes", query=REVIEW_QUERY)
review_infos = get_notes_info(review_note_ids)

review_lemmas = set()
review_lemma_to_note_ids = defaultdict(set)
review_note_lemma = {}
review_field_detected = defaultdict(int)
review_missing_field = 0

for note in review_infos:
    nid = note["noteId"]
    fields = note.get("fields", {})
    lemma_field = resolve_field_name(fields, LEMMA_FIELD_PREFERRED, LEMMA_FIELD_ALIASES)
    if not lemma_field:
        review_missing_field += 1
        continue

    review_field_detected[lemma_field] += 1
    lemma = normalize_lemma(fields[lemma_field].get("value", ""))
    if lemma:
        review_lemmas.add(lemma)
        review_note_lemma[nid] = lemma
        review_lemma_to_note_ids[lemma].add(nid)

print({
    "review_notes_found": len(review_note_ids),
    "review_notes_missing_lemma_field": review_missing_field,
    "review_detected_lemma_fields": dict(review_field_detected),
    "unique_review_lemmas": len(review_lemmas),
})

{'review_notes_found': 867, 'review_notes_missing_lemma_field': 0, 'review_detected_lemma_fields': {'Lemma': 867}, 'unique_review_lemmas': 845}


In [6]:
# 2) Find matching new cards, suspend/tag, and write sibling links
new_card_ids = invoke("findCards", query=NEW_ELIGIBLE_QUERY)
new_cards_info = get_cards_info(new_card_ids)
new_note_ids = sorted({c["note"] for c in new_cards_info})
new_notes_info = get_notes_info(new_note_ids)

new_note_by_id = {n["noteId"]: n for n in new_notes_info}
review_note_by_id = {n["noteId"]: n for n in review_infos}

cards_to_suspend = []
new_notes_to_tag = set()
review_notes_to_tag = set()
duplicate_lemma_to_new_note_ids = defaultdict(set)
new_note_lemma = {}
new_field_detected = defaultdict(int)
new_missing_field = 0
preview = []

for c in new_cards_info:
    cid = c["cardId"]
    nid = c["note"]
    note = new_note_by_id.get(nid)
    if not note:
        continue

    fields = note.get("fields", {})
    lemma_field = resolve_field_name(fields, LEMMA_FIELD_PREFERRED, LEMMA_FIELD_ALIASES)
    if not lemma_field:
        new_missing_field += 1
        continue

    new_field_detected[lemma_field] += 1
    lemma = normalize_lemma(fields[lemma_field].get("value", ""))

    if lemma and lemma in review_lemmas:
        new_note_lemma[nid] = lemma
        cards_to_suspend.append(cid)
        new_notes_to_tag.add(nid)
        duplicate_lemma_to_new_note_ids[lemma].add(nid)
        review_notes_to_tag.update(review_lemma_to_note_ids.get(lemma, set()))

        if len(preview) < 20:
            preview.append({
                "cardId": cid,
                "noteId": nid,
                "lemma": lemma,
            })

cards_to_suspend = sorted(set(cards_to_suspend))
new_notes_to_tag = sorted(new_notes_to_tag)
review_notes_to_tag = sorted(review_notes_to_tag)

# Ensure Link field exists for note types involved in affected notes
affected_note_ids = sorted(set(new_notes_to_tag) | set(review_notes_to_tag))
affected_notes = [new_note_by_id.get(nid) or review_note_by_id.get(nid) for nid in affected_note_ids]
affected_models = sorted({n.get("modelName") for n in affected_notes if n and n.get("modelName")})
models_missing_link = []

for model in affected_models:
    field_names = invoke("modelFieldNames", modelName=model)
    if LINK_FIELD not in field_names:
        models_missing_link.append(model)
        if not DRY_RUN:
            invoke("modelFieldAdd", modelName=model, fieldName=LINK_FIELD)

# Build link updates for sibling groups sharing same lemma
link_updates = []
groups_linked = 0

for lemma, new_ids in duplicate_lemma_to_new_note_ids.items():
    member_ids = sorted(set(new_ids) | set(review_lemma_to_note_ids.get(lemma, set())))
    if len(member_ids) < 2:
        continue

    groups_linked += 1
    member_titles = {}

    for idx, target_nid in enumerate(member_ids, start=1):
        target_note = new_note_by_id.get(target_nid) or review_note_by_id.get(target_nid)
        target_fields = target_note.get("fields", {}) if target_note else {}
        subtitle_field = resolve_field_name(target_fields, SUBTITLE_FIELD_PREFERRED, SUBTITLE_FIELD_ALIASES)
        subtitle = normalize_subtitle(target_fields[subtitle_field].get("value", "")) if subtitle_field else ""

        title = f"{lemma}_{idx}"
        if subtitle:
            title = f"{title}: {subtitle}"
        member_titles[target_nid] = title

    for source_nid in member_ids:
        source_note = new_note_by_id.get(source_nid) or review_note_by_id.get(source_nid)
        source_fields = source_note.get("fields", {}) if source_note else {}
        link_field_name = resolve_field_name(source_fields, LINK_FIELD, [LINK_FIELD.lower()]) or LINK_FIELD

        links = []
        for target_nid in member_ids:
            if target_nid == source_nid:
                continue
            links.append(make_link(member_titles[target_nid], target_nid))

        link_updates.append({"id": source_nid, "fields": {link_field_name: "<br>".join(links)}})

print({
    "new_cards_found": len(new_card_ids),
    "new_notes_found": len(new_note_ids),
    "new_notes_missing_lemma_field": new_missing_field,
    "new_detected_lemma_fields": dict(new_field_detected),
    "cards_to_suspend": len(cards_to_suspend),
    "new_notes_to_tag": len(new_notes_to_tag),
    "review_notes_to_tag": len(review_notes_to_tag),
    "new_note_tag": TAG_SUSPENDED_DUPLICATE,
    "review_note_tag": TAG_UNIQUE_REVIEW_NOTE,
    "affected_note_types": len(affected_models),
    "note_types_missing_link_field": models_missing_link,
    "sibling_groups_linked": groups_linked,
    "notes_with_link_updates": len(link_updates),
    "dry_run": DRY_RUN,
})

print("Preview (first 20 matches):")
for row in preview:
    print(row)

if not DRY_RUN:
    for batch in chunked(cards_to_suspend, 500):
        invoke("suspend", cards=batch)
    for batch in chunked(new_notes_to_tag, 500):
        invoke("addTags", notes=batch, tags=TAG_SUSPENDED_DUPLICATE)
    for batch in chunked(review_notes_to_tag, 500):
        invoke("addTags", notes=batch, tags=TAG_UNIQUE_REVIEW_NOTE)
    for update in link_updates:
        invoke("updateNoteFields", note=update)
    print("Applied suspend + tags + link updates.")
else:
    print("DRY_RUN=True, no changes written.")

{'new_cards_found': 4256, 'new_notes_found': 4239, 'new_notes_missing_lemma_field': 0, 'new_detected_lemma_fields': {'Lemma': 4256}, 'cards_to_suspend': 50, 'new_notes_to_tag': 50, 'review_notes_to_tag': 38, 'new_note_tag': '~suspended_duplicate_lemma_review', 'review_note_tag': '~unique_note_with_suspended_siblings', 'affected_note_types': 1, 'note_types_missing_link_field': [], 'sibling_groups_linked': 27, 'notes_with_link_updates': 71, 'dry_run': False}
Preview (first 20 matches):
{'cardId': 1734681071376, 'noteId': 1734681071376, 'lemma': '確率'}
{'cardId': 1734681071403, 'noteId': 1734681071403, 'lemma': 'ほぼ'}
{'cardId': 1734681071409, 'noteId': 1734681071409, 'lemma': '方向'}
{'cardId': 1734681071412, 'noteId': 1734681071412, 'lemma': 'において'}
{'cardId': 1734681071423, 'noteId': 1734681071423, 'lemma': '幹部'}
{'cardId': 1734681071444, 'noteId': 1734681071444, 'lemma': 'なんとか'}
{'cardId': 1734934742344, 'noteId': 1734510465490, 'lemma': 'とっとと'}
{'cardId': 1737094841496, 'noteId': 1734510

## Apply

1. Run all cells with `DRY_RUN = True`.
2. Verify `cards_to_suspend` and `notes_with_link_updates`.
3. Set `DRY_RUN = False` and rerun the last two code cells.

Make an Anki backup before applying.